# This is a code to test the different MLP performance on data 

## prepared data

In [1]:
# ===== DATA PREPARATION IMPORTS =====
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler


# ===== DATA PREPARATION =====
df = pd.read_csv('datasets/ip_entropies_with_kmeans.csv')
df = df.drop(columns=['src_ip'])

# normalize features
df_normalized = df[['total_sessions', 'username_entropy', 'password_entropy', 'command_entropy']]
df_normalized = df_normalized.fillna(0)

scaler = MinMaxScaler()
normalized_values = scaler.fit_transform(df_normalized)
normalized_data = pd.DataFrame(normalized_values, columns=df_normalized.columns)

# ADD LABELS
normalized_data['cluster'] = df['kmeans_cluster']

# 🔥 FIX IS HERE — reindex BEFORE creating X and y
normalized_data['cluster'] = normalized_data['cluster'].astype('category').cat.codes

# Now create X and y AFTER labels are fixed
X = normalized_data[['total_sessions', 'username_entropy', 'password_entropy', 'command_entropy']]
y = normalized_data[['cluster']]

# Now split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert y to 1D arrays
y_train_1d = y_train.values.ravel()
y_test_1d  = y_test.values.ravel()


## Sci-kit learn MLP

In [2]:
# start scikit learn code here

# ===== SCIKIT-LEARN IMPORTS =====
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report


# ===== SCIKIT-LEARN MLP =====
print("\n===== SCIKIT-LEARN MLP RESULTS =====")

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42
)

mlp.fit(X_train, y_train_1d)
sk_preds = mlp.predict(X_test)

print(classification_report(y_test_1d, sk_preds))
 



===== SCIKIT-LEARN MLP RESULTS =====
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       406
           1       1.00      1.00      1.00       536
           2       1.00      0.97      0.98        29
           3       0.98      1.00      0.99        45

    accuracy                           1.00      1016
   macro avg       0.99      0.99      0.99      1016
weighted avg       1.00      1.00      1.00      1016



## Pytorch 

In [ ]:
# pytorch begins here

# ================================
# PYTORCH SECTION (UPDATED + FIXED)
# ================================

# ----- Imports only for PyTorch section -----
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report


# ----- Convert X and y to tensors -----
X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
X_test_t  = torch.tensor(X_test.values, dtype=torch.float32)

# IMPORTANT: y must be long dtype & 1D
y_train_t = torch.tensor(y_train_1d, dtype=torch.long)
y_test_t  = torch.tensor(y_test_1d, dtype=torch.long)


# ----- Determine sizes -----
input_dim = X_train.shape[1]        # should be 4
num_classes = int(y['cluster'].nunique())  # guaranteed correct after .cat.codes

print("PyTorch Model - Number of classes:", num_classes)


# ----- Define the MLP model -----
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)  # matches exactly the number of classes
        )

    def forward(self, x):
        return self.net(x)


model = MLP(input_dim, num_classes)


# ----- Loss and optimizer -----
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


# ===== TRAINING LOOP =====
epochs = 100
print("\n===== PYTORCH TRAINING =====")
for epoch in range(epochs):
    optimizer.zero_grad()
    
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}")


# ===== EVALUATION =====
model.eval()
with torch.no_grad():
    preds = torch.argmax(model(X_test_t), dim=1)

print("\n===== PYTORCH CLASSIFICATION REPORT =====")
print(classification_report(y_test_t, preds))


PyTorch Model - Number of classes: 4

===== PYTORCH TRAINING =====
Epoch 10/100 - Loss: 1.3917
Epoch 20/100 - Loss: 1.3566
Epoch 30/100 - Loss: 1.3035
Epoch 40/100 - Loss: 1.2329
Epoch 50/100 - Loss: 1.1439
Epoch 60/100 - Loss: 1.0426
Epoch 70/100 - Loss: 0.9464
Epoch 80/100 - Loss: 0.8680
Epoch 90/100 - Loss: 0.7956
Epoch 100/100 - Loss: 0.7134

===== PYTORCH CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0       0.74      0.32      0.44       406
           1       0.64      1.00      0.78       536
           2       0.00      0.00      0.00        29
           3       0.00      0.00      0.00        45

    accuracy                           0.65      1016
   macro avg       0.34      0.33      0.31      1016
weighted avg       0.63      0.65      0.59      1016



/home/savage/Cowrie-Behavioral-Analyasis/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/savage/Cowrie-Behavioral-Analyasis/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/savage/Cowrie-Behavioral-Analyasis/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

: 

## TensorFlow

In [ ]:
# ================================
# TENSORFLOW SECTION (FIXED)
# ================================

# ----- Imports for TensorFlow section -----
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report


# ----- Verify X_train exists -----
print("X_train shape:", X_train.shape)
print("y_train_1d shape:", y_train_1d.shape)

input_dim = X_train.shape[1]      # should be 4
num_classes = int(y['cluster'].nunique())

print("TensorFlow Model - Number of classes:", num_classes)


# ----- Build the Keras model -----
model_tf = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model_tf.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


# ----- Train the model -----
history = model_tf.fit(
    X_train, y_train_1d,
    validation_data=(X_test, y_test_1d),
    epochs=50,
    batch_size=32,
    verbose=1
)


# ----- Evaluate the model -----
tf_preds = model_tf.predict(X_test).argmax(axis=1)

print("\n===== TENSORFLOW CLASSIFICATION REPORT =====")
print(classification_report(y_test_1d, tf_preds))


2025-12-11 17:03:55.929326: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-11 17:03:56.398262: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-11 17:03:58.515263: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


X_train shape: (4061, 4)
y_train_1d shape: (4061,)
TensorFlow Model - Number of classes: 4


I0000 00:00:1765490639.699854    6592 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4085 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
2025-12-11 17:04:00.010053: W external/local_xla/xla/service/gpu/llvm_gpu_backend/default/nvptx_libdevice_path.cc:41] Can't find libdevice directory ${CUDA_DIR}/nvvm/libdevice. This may result in compilation or runtime failures, if the program we try to run uses routines from libdevice.
Searched for CUDA in the following directories:
  ./cuda_sdk_lib
  ipykernel_launcher.runfiles/cuda_nvcc
  ipykernel_launcher.runfiles/cuda_nvdisasm
  ipykernel_launcher.runfiles/nvidia_nvshmem
  ipykern/cuda_nvcc
  ipykern/cuda_nvdisasm
  ipykern/nvidia_nvshmem
  
  /usr/local/cuda
  /opt/cuda
  /home/savage/Cowrie-Behavioral-Analyasis/venv/lib/python3.10/site-packages/tensorflow/python/platform/../../../nvidia/cuda_nvcc
  /home/savage/Cowrie-Behavioral-Analyas

Epoch 1/50


2025-12-11 17:04:01.203953: I external/local_xla/xla/service/service.cc:163] XLA service 0x74b2f800b030 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-12-11 17:04:01.203967: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6
2025-12-11 17:04:01.218728: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
